## 📚 **1. IMPORTACIÓN Y CONFIGURACIÓN**

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
from datetime import datetime

# Librerías para modelado
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import optuna

# Configuraciones
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Librerías importadas correctamente")
print(f"📅 Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 📂 **2. CARGA DE DATOS PREPARADOS**

In [ ]:
# Cargar datos del análisis exploratorio
try:
    with open('datasets_preparados.pkl', 'rb') as f:
        datasets = pickle.load(f)
    
    # Extraer datasets
    data = datasets['data']
    train_to1 = datasets['train_to1']
    train_to2 = datasets['train_to2']
    test_t1 = datasets['test_t1']
    test_to2 = datasets['test_to2']
    train_len = datasets['train_len']
    
    print("✅ Datos cargados exitosamente desde análisis exploratorio")
    print(f"📊 Configuración: {train_len} train / {len(data) - train_len} test")
    
except FileNotFoundError:
    print("❌ Error: No se encontró 'datasets_preparados.pkl'")
    print("💡 Ejecuta primero el notebook '01_Analisis_Exploratorio.ipynb'")

## 📈 **3. FUNCIONES DE EVALUACIÓN**

In [ ]:
# ==================== FUNCIONES PARA MEDIA MÓVIL ====================

def evaluar_media_movil_rolling(train_data, test_data, ventana):
    """
    Evalúa media móvil usando ventana móvil (walking forward)
    Predice solo 1 paso adelante en cada iteración usando datos reales disponibles
    """
    predicciones = []
    datos_disponibles = train_data.copy()
    
    for i in range(len(test_data)):
        # Predecir solo el siguiente período usando los últimos 'ventana' datos reales
        prediccion = datos_disponibles.iloc[-ventana:].mean().values[0]
        predicciones.append(prediccion)
        
        # Agregar el dato real del test para la siguiente predicción
        valor_real = test_data.iloc[i].values[0]
        nuevo_indice = datos_disponibles.index[-1] + 1
        datos_disponibles.loc[nuevo_indice] = valor_real
    
    return pd.Series(predicciones, index=test_data.index)


def pronosticar_media_movil(datos, ventana, horizonte):
    """
    Genera pronósticos futuros usando media móvil
    """
    data = datos.copy()
    
    for i in range(horizonte):
        nuevo_valor = data.iloc[-ventana:].mean().values[0]
        nuevo_indice = data.index[-1] + 1
        data.loc[nuevo_indice] = nuevo_valor
    
    return data.iloc[-horizonte:]


print("✅ Funciones de Media Móvil definidas")

In [ ]:
# ==================== FUNCIONES PARA HOLT-WINTERS ====================

def evaluar_rolling_forecast_holtwinters(serie, alpha, beta, gamma, seasonal=None, trend='add',
                                          window_size=60, window=6, step_size=1, 
                                          horizon=1, metric='rmse'):
    """Evalúa Holt-Winters con validación temporal"""
    serie = pd.Series(serie).astype(float).dropna()
    n = len(serie)
    predichos, observados = [], []
    
    puntos_finales = list(range(n - window * step_size, n - horizon + 1, step_size))
    
    for end_train in puntos_finales:
        start_train = max(0, end_train - window_size)
        train = serie.iloc[start_train:end_train]
        test = serie.iloc[end_train:end_train + horizon]
        
        if len(test) < horizon or len(train) < 12:
            continue
            
        try:
            if seasonal is None:
                # Solo tendencia
                model = ExponentialSmoothing(
                    train, trend=trend, seasonal=None
                ).fit(smoothing_level=alpha, smoothing_trend=beta, optimized=False)
            else:
                # Con estacionalidad
                seasonal_periods = min(6, len(train)//2)
                model = ExponentialSmoothing(
                    train, trend=trend, seasonal=seasonal, 
                    seasonal_periods=seasonal_periods
                ).fit(smoothing_level=alpha, smoothing_trend=beta, 
                      smoothing_seasonal=gamma, optimized=False)
            
            pred = model.forecast(steps=horizon)
            if not np.isfinite(pred).all():
                continue
                
            predichos.extend(pred.tolist())
            observados.extend(test.tolist())
                
        except:
            continue
            
    if len(predichos) == 0:
        return np.inf
    
    predichos = np.array(predichos)
    observados = np.array(observados)
    
    if not np.isfinite(predichos).all() or not np.isfinite(observados).all():
        return np.inf
    
    if metric == 'rmse':
        return np.sqrt(mean_squared_error(observados, predichos))
    else:
        return np.mean(np.abs(observados - predichos))


print("✅ Funciones de Holt-Winters definidas")

## 📊 **4. MEDIA MÓVIL - PRODUCTO 1**

In [ ]:
# Evaluar diferentes ventanas para Media Móvil - Producto 1
print("📈 EVALUACIÓN MEDIA MÓVIL - PRODUCTO 1")
print("=" * 50)

ventanas = [3, 6, 9, 12, 15, 18, 21, 24]
resultados_ma_p1 = {}

for ventana in ventanas:
    pred_rolling = evaluar_media_movil_rolling(train_to1['producto1'], test_t1['producto1'], ventana)
    rmse_rolling = np.sqrt(mean_squared_error(test_t1['producto1'], pred_rolling))
    resultados_ma_p1[ventana] = {
        'rmse': rmse_rolling,
        'predicciones': pred_rolling
    }
    print(f"Ventana {ventana:2d}: RMSE = {rmse_rolling:.4f}")

# Encontrar mejor ventana
mejor_ventana_ma_p1 = min(resultados_ma_p1.keys(), key=lambda x: resultados_ma_p1[x]['rmse'])
print(f"\n🏆 Mejor ventana: {mejor_ventana_ma_p1} (RMSE: {resultados_ma_p1[mejor_ventana_ma_p1]['rmse']:.4f})")

# Visualizar resultados
plt.figure(figsize=(12, 8))

# Datos de entrenamiento y test
plt.plot(train_to1.index, train_to1['producto1'], label='Entrenamiento', color='steelblue', alpha=0.8, linewidth=2)
plt.plot(test_t1.index, test_t1['producto1'], label='Real (Test)', color='red', alpha=0.8, linewidth=2)

# Predicción con mejor ventana
mejor_pred = resultados_ma_p1[mejor_ventana_ma_p1]['predicciones']
plt.plot(test_t1.index, mejor_pred, label=f'Predicción MA({mejor_ventana_ma_p1})', 
         color='orange', alpha=0.8, linewidth=2, linestyle='--')

plt.axvline(x=train_len-1, color='black', linestyle='-', alpha=0.5, linewidth=1)
plt.title(f'📈 Media Móvil - Producto 1 | Mejor Ventana: {mejor_ventana_ma_p1}', fontsize=14, fontweight='bold')
plt.xlabel('Tiempo')
plt.ylabel('Valores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Gráfico de RMSE por ventana
plt.figure(figsize=(10, 6))
rmse_values = [resultados_ma_p1[v]['rmse'] for v in ventanas]
plt.plot(ventanas, rmse_values, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=min(rmse_values), color='red', linestyle='--', alpha=0.7, label=f'Mejor RMSE: {min(rmse_values):.4f}')
plt.xlabel('Tamaño de Ventana')
plt.ylabel('RMSE')
plt.title('📊 Rendimiento Media Móvil por Tamaño de Ventana - Producto 1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔄 **5. HOLT-WINTERS - PRODUCTO 1**

In [ ]:
# Optimización Holt-Winters para Producto 1
def objective_hw_p1(trial):
    """Función objetivo para optimización de Holt-Winters - Producto 1"""
    alpha = trial.suggest_float("alpha", 0.001, 0.3)
    beta = trial.suggest_float("beta", 0.001, 0.3)
    gamma = trial.suggest_float("gamma", 0.001, 0.3)
    seasonal = trial.suggest_categorical("seasonal", ["add", "mul", None])
    trend = trial.suggest_categorical("trend", ["add", "mul", None])
    window_size = trial.suggest_int("window_size", 24, 72)
    
    return evaluar_rolling_forecast_holtwinters(
        train_to1["producto1"], alpha, beta, gamma,
        seasonal=seasonal,
        trend=trend,
        window_size=window_size,
        window=6,
        step_size=1,
        horizon=1,
        metric='rmse'
    )

print("🔄 Iniciando optimización Holt-Winters para Producto 1...")
print("⏱️ Esto puede tomar varios minutos...")
print("-" * 60)

study_hw_p1 = optuna.create_study(direction="minimize")
study_hw_p1.optimize(objective_hw_p1, n_trials=100)

print("\n" + "=" * 60)
print("RESULTADOS OPTIMIZACIÓN HOLT-WINTERS - PRODUCTO 1")
print("=" * 60)
print(f"Mejor RMSE: {study_hw_p1.best_value:.4f}")
print(f"\nMejores parámetros:")
for param, value in study_hw_p1.best_params.items():
    if isinstance(value, float):
        print(f"  {param}: {value:.4f}")
    else:
        print(f"  {param}: {value}")
print("=" * 60)

## 📊 **6. MEDIA MÓVIL - PRODUCTO 2**

In [ ]:
# Evaluar diferentes ventanas para Media Móvil - Producto 2
print("📈 EVALUACIÓN MEDIA MÓVIL - PRODUCTO 2")
print("=" * 50)

resultados_ma_p2 = {}

for ventana in ventanas:
    pred_rolling = evaluar_media_movil_rolling(train_to2['producto2'], test_to2['producto2'], ventana)
    rmse_rolling = np.sqrt(mean_squared_error(test_to2['producto2'], pred_rolling))
    resultados_ma_p2[ventana] = {
        'rmse': rmse_rolling,
        'predicciones': pred_rolling
    }
    print(f"Ventana {ventana:2d}: RMSE = {rmse_rolling:.4f}")

# Encontrar mejor ventana
mejor_ventana_ma_p2 = min(resultados_ma_p2.keys(), key=lambda x: resultados_ma_p2[x]['rmse'])
print(f"\n🏆 Mejor ventana: {mejor_ventana_ma_p2} (RMSE: {resultados_ma_p2[mejor_ventana_ma_p2]['rmse']:.4f})")

# Visualizar resultados
plt.figure(figsize=(12, 8))

plt.plot(train_to2.index, train_to2['producto2'], label='Entrenamiento', color='steelblue', alpha=0.8, linewidth=2)
plt.plot(test_to2.index, test_to2['producto2'], label='Real (Test)', color='red', alpha=0.8, linewidth=2)

mejor_pred_p2 = resultados_ma_p2[mejor_ventana_ma_p2]['predicciones']
plt.plot(test_to2.index, mejor_pred_p2, label=f'Predicción MA({mejor_ventana_ma_p2})', 
         color='orange', alpha=0.8, linewidth=2, linestyle='--')

plt.axvline(x=train_len-1, color='black', linestyle='-', alpha=0.5, linewidth=1)
plt.title(f'📈 Media Móvil - Producto 2 | Mejor Ventana: {mejor_ventana_ma_p2}', fontsize=14, fontweight='bold')
plt.xlabel('Tiempo')
plt.ylabel('Valores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔄 **7. HOLT-WINTERS - PRODUCTO 2**

In [ ]:
# Optimización Holt-Winters para Producto 2
def objective_hw_p2(trial):
    """Función objetivo para optimización de Holt-Winters - Producto 2"""
    alpha = trial.suggest_float("alpha", 0.001, 0.3)
    beta = trial.suggest_float("beta", 0.001, 0.3)
    gamma = trial.suggest_float("gamma", 0.001, 0.3)
    seasonal = trial.suggest_categorical("seasonal", ["add", "mul", None])
    trend = trial.suggest_categorical("trend", ["add", "mul", None])
    window_size = trial.suggest_int("window_size", 24, 72)
    
    return evaluar_rolling_forecast_holtwinters(
        train_to2["producto2"], alpha, beta, gamma,
        seasonal=seasonal,
        trend=trend,
        window_size=window_size,
        window=6,
        step_size=1,
        horizon=1,
        metric='rmse'
    )

print("🔄 Iniciando optimización Holt-Winters para Producto 2...")
print("⏱️ Esto puede tomar varios minutos...")
print("-" * 60)

study_hw_p2 = optuna.create_study(direction="minimize")
study_hw_p2.optimize(objective_hw_p2, n_trials=100)

print("\n" + "=" * 60)
print("RESULTADOS OPTIMIZACIÓN HOLT-WINTERS - PRODUCTO 2")
print("=" * 60)
print(f"Mejor RMSE: {study_hw_p2.best_value:.4f}")
print(f"\nMejores parámetros:")
for param, value in study_hw_p2.best_params.items():
    if isinstance(value, float):
        print(f"  {param}: {value:.4f}")
    else:
        print(f"  {param}: {value}")
print("=" * 60)

## 🔮 **8. PRONÓSTICOS FUTUROS**

In [ ]:
# Generar pronósticos futuros para ambos productos
horizonte_pronostico = 12  # 12 períodos futuros

print(f"🔮 GENERANDO PRONÓSTICOS FUTUROS ({horizonte_pronostico} períodos)")
print("=" * 60)

# Pronósticos Media Móvil
print("📈 Pronósticos con Media Móvil:")

# Producto 1 - Media Móvil
datos_completos_p1 = pd.concat([train_to1['producto1'], test_t1['producto1']])
pronostico_ma_p1 = pronosticar_media_movil(datos_completos_p1, mejor_ventana_ma_p1, horizonte_pronostico)
print(f"  Producto 1 (MA {mejor_ventana_ma_p1}): {pronostico_ma_p1.iloc[-1]:.2f} (último período)")

# Producto 2 - Media Móvil  
datos_completos_p2 = pd.concat([train_to2['producto2'], test_to2['producto2']])
pronostico_ma_p2 = pronosticar_media_movil(datos_completos_p2, mejor_ventana_ma_p2, horizonte_pronostico)
print(f"  Producto 2 (MA {mejor_ventana_ma_p2}): {pronostico_ma_p2.iloc[-1]:.2f} (último período)")

# Pronósticos Holt-Winters
print("\n🔄 Pronósticos con Holt-Winters:")

# Producto 1 - Holt-Winters
try:
    best_params_p1 = study_hw_p1.best_params
    
    if best_params_p1['seasonal'] is None:
        model_hw_p1 = ExponentialSmoothing(
            datos_completos_p1, 
            trend=best_params_p1['trend'], 
            seasonal=None
        ).fit(
            smoothing_level=best_params_p1['alpha'], 
            smoothing_trend=best_params_p1['beta'], 
            optimized=False
        )
    else:
        model_hw_p1 = ExponentialSmoothing(
            datos_completos_p1, 
            trend=best_params_p1['trend'], 
            seasonal=best_params_p1['seasonal'],
            seasonal_periods=6
        ).fit(
            smoothing_level=best_params_p1['alpha'], 
            smoothing_trend=best_params_p1['beta'],
            smoothing_seasonal=best_params_p1['gamma'], 
            optimized=False
        )
    
    pronostico_hw_p1 = model_hw_p1.forecast(steps=horizonte_pronostico)
    print(f"  Producto 1 (HW): {pronostico_hw_p1.iloc[-1]:.2f} (último período)")
    
except Exception as e:
    print(f"  ⚠️ Error en pronóstico HW Producto 1: {str(e)}")
    pronostico_hw_p1 = None

# Producto 2 - Holt-Winters
try:
    best_params_p2 = study_hw_p2.best_params
    
    if best_params_p2['seasonal'] is None:
        model_hw_p2 = ExponentialSmoothing(
            datos_completos_p2, 
            trend=best_params_p2['trend'], 
            seasonal=None
        ).fit(
            smoothing_level=best_params_p2['alpha'], 
            smoothing_trend=best_params_p2['beta'], 
            optimized=False
        )
    else:
        model_hw_p2 = ExponentialSmoothing(
            datos_completos_p2, 
            trend=best_params_p2['trend'], 
            seasonal=best_params_p2['seasonal'],
            seasonal_periods=6
        ).fit(
            smoothing_level=best_params_p2['alpha'], 
            smoothing_trend=best_params_p2['beta'],
            smoothing_seasonal=best_params_p2['gamma'], 
            optimized=False
        )
    
    pronostico_hw_p2 = model_hw_p2.forecast(steps=horizonte_pronostico)
    print(f"  Producto 2 (HW): {pronostico_hw_p2.iloc[-1]:.2f} (último período)")
    
except Exception as e:
    print(f"  ⚠️ Error en pronóstico HW Producto 2: {str(e)}")
    pronostico_hw_p2 = None

In [ ]:
# Visualizar pronósticos
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))

# Producto 1
ax1.plot(datos_completos_p1.index, datos_completos_p1, label='Datos Históricos', color='steelblue', linewidth=2)
ax1.plot(pronostico_ma_p1.index, pronostico_ma_p1, label=f'Media Móvil ({mejor_ventana_ma_p1})', 
         color='orange', linewidth=2, linestyle='--', marker='o', markersize=4)

if pronostico_hw_p1 is not None:
    future_index_p1 = range(len(datos_completos_p1), len(datos_completos_p1) + horizonte_pronostico)
    ax1.plot(future_index_p1, pronostico_hw_p1, label='Holt-Winters', 
             color='green', linewidth=2, linestyle=':', marker='s', markersize=4)

ax1.axvline(x=len(datos_completos_p1)-1, color='red', linestyle='-', alpha=0.7, linewidth=2, label='Inicio Pronóstico')
ax1.set_title('🔮 Pronósticos Futuros - Producto 1', fontsize=14, fontweight='bold')
ax1.set_ylabel('Valores')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Producto 2
ax2.plot(datos_completos_p2.index, datos_completos_p2, label='Datos Históricos', color='steelblue', linewidth=2)
ax2.plot(pronostico_ma_p2.index, pronostico_ma_p2, label=f'Media Móvil ({mejor_ventana_ma_p2})', 
         color='orange', linewidth=2, linestyle='--', marker='o', markersize=4)

if pronostico_hw_p2 is not None:
    future_index_p2 = range(len(datos_completos_p2), len(datos_completos_p2) + horizonte_pronostico)
    ax2.plot(future_index_p2, pronostico_hw_p2, label='Holt-Winters', 
             color='green', linewidth=2, linestyle=':', marker='s', markersize=4)

ax2.axvline(x=len(datos_completos_p2)-1, color='red', linestyle='-', alpha=0.7, linewidth=2, label='Inicio Pronóstico')
ax2.set_title('🔮 Pronósticos Futuros - Producto 2', fontsize=14, fontweight='bold')
ax2.set_ylabel('Valores')
ax2.set_xlabel('Tiempo')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 **9. RESUMEN COMPARATIVO**

In [ ]:
# Tabla comparativa de resultados
print("📊 RESUMEN COMPARATIVO DE MODELOS")
print("=" * 70)

# Crear tabla de resultados
resultados_resumen = {
    'Modelo': [],
    'Producto 1 RMSE': [],
    'Producto 2 RMSE': []
}

# Media Móvil
resultados_resumen['Modelo'].append(f'Media Móvil ({mejor_ventana_ma_p1}/{mejor_ventana_ma_p2})')
resultados_resumen['Producto 1 RMSE'].append(resultados_ma_p1[mejor_ventana_ma_p1]['rmse'])
resultados_resumen['Producto 2 RMSE'].append(resultados_ma_p2[mejor_ventana_ma_p2]['rmse'])

# Holt-Winters
resultados_resumen['Modelo'].append('Holt-Winters')
resultados_resumen['Producto 1 RMSE'].append(study_hw_p1.best_value)
resultados_resumen['Producto 2 RMSE'].append(study_hw_p2.best_value)

df_resumen = pd.DataFrame(resultados_resumen)
print(df_resumen.to_string(index=False))

# Mejor modelo por producto
print(f"\n🏆 MEJORES MODELOS:")
print(f"   • Producto 1: {'Media Móvil' if resultados_ma_p1[mejor_ventana_ma_p1]['rmse'] < study_hw_p1.best_value else 'Holt-Winters'}")
print(f"   • Producto 2: {'Media Móvil' if resultados_ma_p2[mejor_ventana_ma_p2]['rmse'] < study_hw_p2.best_value else 'Holt-Winters'}")

# Guardar resultados
resultados_ma_hw = {
    'ma_p1': resultados_ma_p1,
    'ma_p2': resultados_ma_p2,
    'hw_p1': {
        'study': study_hw_p1,
        'best_params': study_hw_p1.best_params,
        'best_rmse': study_hw_p1.best_value
    },
    'hw_p2': {
        'study': study_hw_p2,
        'best_params': study_hw_p2.best_params,
        'best_rmse': study_hw_p2.best_value
    },
    'pronosticos': {
        'ma_p1': pronostico_ma_p1,
        'ma_p2': pronostico_ma_p2,
        'hw_p1': pronostico_hw_p1 if 'pronostico_hw_p1' in locals() else None,
        'hw_p2': pronostico_hw_p2 if 'pronostico_hw_p2' in locals() else None
    }
}

with open('resultados_ma_hw.pkl', 'wb') as f:
    pickle.dump(resultados_ma_hw, f)

print("\n💾 Resultados guardados en 'resultados_ma_hw.pkl'")
print("\n" + "=" * 70)
print("🚀 ANÁLISIS MEDIA MÓVIL Y HOLT-WINTERS COMPLETADO")
print("📝 Continúa con el notebook '03_ARIMA.ipynb'")
print("=" * 70)